In [54]:
import os, glob
import pandas as pd
import numpy as np

from matplotlib import pyplot as plt


In [55]:
data_dir = r"D:\!Work\IFA\!data\Urban_fluxes_NE\processed"

city_name = 'Tomsk'
corr_name = 'new_rot3'

# city_name = 'Beijing'
# corr_name = ''

# city_name = 'Moscow'
# corr_name = ''


df4sites = {}

data_dir_final = data_dir + '/' + city_name + ('_' if corr_name else '') + corr_name

file_pattern =  data_dir_final + '/*.csv'
display(file_pattern)

for file in glob.glob(file_pattern):
    site = os.path.basename(file).split('.')[0]
    print (file)
    df = pd.read_csv(file, index_col=0, parse_dates=True)
    df4sites[site] = df

'D:\\!Work\\IFA\\!data\\Urban_fluxes_NE\\processed/Tomsk_new_rot3/*.csv'

D:\!Work\IFA\!data\Urban_fluxes_NE\processed/Tomsk_new_rot3\BEK.csv
D:\!Work\IFA\!data\Urban_fluxes_NE\processed/Tomsk_new_rot3\IMCES30m.csv
D:\!Work\IFA\!data\Urban_fluxes_NE\processed/Tomsk_new_rot3\IMCES30m_mezo.csv
D:\!Work\IFA\!data\Urban_fluxes_NE\processed/Tomsk_new_rot3\IMCES40m.csv
D:\!Work\IFA\!data\Urban_fluxes_NE\processed/Tomsk_new_rot3\Oblkom.csv


In [61]:
#filter_vars = ['Qh', 'Qtau'] #, 'Cd']
filter_lims = {'Qh': [-300, 600], 'Qle': [-300, 600], 'Qtau': [0, 5], 'Cd': [0, 1]}
filter_sigmas = {'Qh': 5, 'Qle': 5, 'Qtau': 5, 'Cd': 6}

def apply_qc (df, filter_vars, qc_ext_window = '1h', outl_stat_window = '60D', n_sigma = 5, use_hourly_stats = True):

    df = df.copy()

    for var in filter_vars.keys():

        cur_n_sigma = n_sigma[var] if isinstance (n_sigma, dict) else n_sigma

        if var not in df.columns: continue  
        
        df[f'qc_{var}_lims'] = np.zeros_like(df[var]).astype(bool)
        
        df[f'qc_{var}_lims'] = (df[var] >= filter_lims[var][0]) & (df[var] <= filter_lims[var][1])
        df.loc[np.isnan(df[var]), f'qc_{var}_lims'] = True
        n_lims = (~df[f'qc_{var}_lims']).sum()
        
        df[f'qc_{var}_outl'] = np.ones_like(df[var]).astype(bool)

        cur_n_outl = 1
        n_outl = []

        while cur_n_outl > 0:

            var_f = df[var].where(df[f'qc_{var}_outl'])
            var_f2stats = df[var].where(df[f'qc_{var}_lims'] & df[f'qc_{var}_outl'])

            if use_hourly_stats:
                outl_lim1 = var_f * np.nan
                outl_lim2 = var_f * np.nan
                for hour in range (0, 24):
                    hours_ext = np.arange(hour-1, hour+2)
                    hours_ext [hours_ext<0] += 24

                    is_hours_est = np.isin (df.index.hour, hours_ext)
                    is_hour = df.index.hour == hour

                    var_rol_mean =  var_f2stats.where(is_hours_est).rolling(outl_stat_window, center=True, min_periods=1).mean()
                    var_rol_std  =  var_f2stats.where(is_hours_est).rolling(outl_stat_window, center=True, min_periods=1).std()
            
                    outl_lim1[is_hour] = (var_rol_mean - cur_n_sigma * var_rol_std)[is_hour]
                    outl_lim2[is_hour] = (var_rol_mean + cur_n_sigma * var_rol_std)[is_hour]
            else:
                var_rol_mean =  var_f2stats.rolling(outl_stat_window, center=True, min_periods=1).mean()
                var_rol_std  =  var_f2stats.rolling(outl_stat_window, center=True, min_periods=1).std()
            
                outl_lim1 = (var_rol_mean - cur_n_sigma * var_rol_std)
                outl_lim2 = (var_rol_mean + cur_n_sigma * var_rol_std)

            df[f'{var}_outl_lim1'] = outl_lim1
            df[f'{var}_outl_lim2'] = outl_lim2

            is_outlier = (var_f < outl_lim1) | (var_f > outl_lim2)
            is_outlier [np.isnan(df[var])] = False
            cur_n_outl = (is_outlier).sum()

            df.loc[is_outlier, f'qc_{var}_outl'] = False

            n_outl.append(cur_n_outl)

            if len(n_outl) > 2 and n_outl[-1] <= 2 and n_outl[-2] <= 2:
                break

        print (f'{var}: {n_lims}, {n_outl}')
        df[f'qc_{var}_sum'] = df[[c for c in df.columns if f'qc_{var}' in c]].astype(float).sum(axis=1)
    

    df[f'qc_tot'] = (df[[f'qc_{var}_sum' for var in filter_vars.keys() if var in df.columns]] >= 2).min(axis=1) #in filter_vars.keys()]
    df[f'qc_tot_noCd'] = (df[[f'qc_{var}_sum' for var in filter_vars.keys() if var in df.columns if 'Cd' not in var]] >= 2).min(axis=1) #in filter_vars.keys()]

    print (f'total: { (~df[f'qc_tot']).sum()}, total_noCd: {(~df[f'qc_tot_noCd']).sum()}')

    df[f'qc_tot_ext'] = df[f'qc_tot'].rolling(qc_ext_window, center=True).min().astype(bool)        
        
    
    return df

df4sites_qc_d = {}
df4sites_qc_h = {}

for site in df4sites.keys():
    df4sites[site]['Cd'] = (df4sites[site]['Ustar'] / df4sites[site]['Wind_vel']) **2
    #df4sites_qc_d[site] = apply_qc(df4sites[site], filter_lims, use_hourly_stats=False)
    print (f'*** starting QC for {site} ***')
    if site != 'ERA5':
        df4sites_qc_h[site] = apply_qc(df4sites[site], filter_lims, use_hourly_stats=True, n_sigma=filter_sigmas)
    else:
        df4sites_qc_h[site] = df4sites[site].copy()
    #ak



*** starting QC for BEK ***
Qh: 31, [106, 33, 11, 1, 0]
Qtau: 5, [48, 20, 9, 2, 2]
Cd: 31, [120, 57, 31, 12, 7, 5, 1, 0]
total: 361, total_noCd: 196
*** starting QC for IMCES30m ***
Qh: 44, [105, 7, 1, 0]
Qtau: 7, [69, 32, 9, 1, 0]
Cd: 20, [100, 40, 21, 7, 0]
total: 303, total_noCd: 201
*** starting QC for IMCES30m_mezo ***
Qh: 80, [129, 13, 5, 1, 0]
Qtau: 24, [127, 42, 11, 2, 1]
Cd: 26, [174, 94, 41, 17, 4, 3, 0]
total: 443, total_noCd: 245
*** starting QC for IMCES40m ***
Qh: 110, [207, 36, 12, 2, 1]
Qtau: 17, [103, 25, 9, 7, 5, 1, 3, 0]
Cd: 37, [172, 65, 27, 4, 0]
total: 536, total_noCd: 373
*** starting QC for Oblkom ***
Qh: 173, [212, 34, 9, 5, 0]
Qtau: 2, [62, 22, 10, 2, 0]
Cd: 61, [172, 50, 21, 7, 4, 4, 4, 1, 1]
total: 476, total_noCd: 344


In [57]:
var2draw = 'Qh'
site2draw = 'BEK'

df2draw = pd.DataFrame(df4sites_qc_h[site2draw][var2draw])

df2draw[var2draw + '_f'] = df2draw[var2draw].where(df4sites_qc_h[site2draw]['qc_tot_ext'])

pd.options.plotting.backend = "plotly"

df2draw.plot()




In [59]:
out_dir = data_dir_final + '/L1'
os.makedirs(out_dir, exist_ok=True)

for site in df4sites_qc_h.keys():

    df4sites_qc_h[site] = df4sites_qc_h[site].drop(columns = [c for c in df4sites_qc_h[site].columns if ('qc' in c and 'tot' not in c) or 'lim' in c])

    for c in [c for c in df4sites_qc_h[site].columns if 'qc_' in c]:
        df4sites_qc_h[site][c] = df4sites_qc_h[site][c].astype(int)
        
    df4sites_qc_h[site].to_csv(f'{out_dir}/{site}.csv')